# Gold volatility diagnostics and forecasts

This notebook starts with ACF/PACF plots for returns and squared returns, then compares seven return-only baselines and a Hyndman-Khandakar ARIMA forecast with the five-session realized-variance target. HK order selection is repeated using only returns available at each origin. The latest 20 common origins keep refitting manageable; overlapping targets and the small window make the results exploratory, not a final model decision.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

cwd = Path.cwd()
repo_root = next(
    (
        candidate
        for candidate in (cwd, *cwd.parents)
        if (candidate / "pyproject.toml").is_file()
        and (candidate / "src" / "gold_vol_pipeline").is_dir()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError(
        f"Could not find the project root above {cwd}. "
        "Run this notebook from the project directory or one of its subdirectories."
    )

sys.path.insert(0, str(repo_root / "src"))

from gold_vol_pipeline import prediction_history
from gold_vol_pipeline.auto_arima import hyndman_khandakar
from gold_vol_pipeline.model_evaluation import plot_acf_pacf

data_path = repo_root / "data" / "processed" / "gold_futures.csv"
if not data_path.is_file():
    raise FileNotFoundError(
        f"Prepared gold data was not found at {data_path}. "
        "Run the data-preparation step first."
    )
plot_dir = repo_root / "plots" / "model_comparisons"
print(f"Using data file: {data_path}")


In [ ]:
df = pd.read_csv(data_path)
df.head()


## ACF/PACF diagnostics

The plots show linear serial dependence in returns and squared returns. They are exploratory diagnostics and do not by themselves establish a model specification or assumption.

In [ ]:
import matplotlib.pyplot as plt

returns = pd.to_numeric(df['log_return'], errors='raise').dropna()
for series_name, series in [
    ('log returns', returns),
    ('squared log returns', returns.pow(2)),
]:
    figure, axes = plot_acf_pacf(series, name=series_name, lags=40)
    display(figure)
    plt.close(figure)


## Rolling-origin model comparison

Each forecast uses data available through its origin. The HK selector is refitted to squared returns at each of the latest 20 origins; only dates where both the baseline forecasts and HK forecast are available are compared.

In [ ]:
comparison_window = 20
comparison = prediction_history(
    str(data_path),
    horizon=5,
    sample_size=comparison_window,
)
comparison.head()


In [ ]:
hk_rows = []
for index in df.tail(comparison_window).index:
    origin_returns = pd.to_numeric(
        df.loc[:index, 'log_return'], errors='raise'
    ).dropna()
    hk_result = hyndman_khandakar(
        origin_returns.pow(2), max_p=5, max_q=5, max_d=2, max_order=5
    )
    hk_rows.append(
        {
            'date': df.loc[index, 'date'],
            'auto_arima': float(hk_result.forecast(horizon=5).sum()),
            'selected_order': f'ARIMA{hk_result.order}',
            'AICc': hk_result.aicc,
        }
    )

hk_results = pd.DataFrame(hk_rows)
comparison = comparison.merge(
    hk_results[['date', 'auto_arima']],
    on='date',
    how='inner',
    validate='one_to_one',
)
if comparison.empty:
    raise RuntimeError('No common forecast origins were available for comparison.')
display(hk_results)
print(f'Common forecast origins: {len(comparison)}')


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display

plot_dir.mkdir(parents=True, exist_ok=True)
model_columns = [
    'historical_variance',
    'rolling_variance',
    'ewma_variance',
    'ar1_squared_return',
    'ma1_squared_return',
    'arch1',
    'garch11',
    'auto_arima',
]
metrics = pd.DataFrame([
    {
        'model': model,
        'MAE': (comparison[model] - comparison['actual']).abs().mean(),
        'RMSE': ((comparison[model] - comparison['actual']) ** 2).mean() ** 0.5,
    }
    for model in model_columns
]).sort_values('RMSE')
display(metrics)

plot_paths = []
for model in model_columns:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(comparison['date'], comparison['actual'], label='actual', color='black', linewidth=1.5)
    ax.plot(comparison['date'], comparison[model], label=f'{model} forecast', alpha=0.85)
    ax.set_title(f'Actual vs {model} forecast (5-session realized variance)')
    ax.set_xlabel('Forecast origin date')
    ax.set_ylabel('Realized variance')
    ax.legend(loc='upper right')
    plt.xticks(rotation=45)
    fig.tight_layout()
    output_path = plot_dir / f'{model}_vs_actual.png'
    fig.savefig(output_path, dpi=200)
    plt.close(fig)
    plot_paths.append(output_path)
    display(Image(filename=str(output_path)))

print('Saved comparison charts:')
for output_path in plot_paths:
    print(output_path)
